In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
from matplotlib.patches import Patch
from statsmodels.stats.multitest import multipletests
from sklearn import svm
from sklearn.metrics import roc_auc_score

In [2]:
expr = pd.read_csv("CGS4144-bioinfo_project/data/expr_by_library.tsv.gz", sep="\t", index_col=0)
meta = pd.read_csv("CGS4144-bioinfo_project/data/meta_by_library.tsv", sep="\t", index_col=0)

In [3]:
sl = meta.index.str.extract(r"SL(\d+)")[0].astype(int).values
meta["batch"] = np.where(sl > 20000, "B", "A")

keep = meta[(meta.region == "ancg")].copy()
keep["group"] = np.where(keep.dx == "control", "Control", "Disorder")

#log scale and filter genes with zero expression (about 8000) before doing plots
log_expr = np.log2(expr[keep.index] + 1)
log_expr = log_expr[log_expr.median(axis=1) > 0]

#select 5000 most variable genes 
top = log_expr.var(axis=1).sort_values(ascending=False).index[:5000]
X = log_expr.loc[top].T

In [4]:
labels=keep['group'].tolist()

#extract clustering labels from assignment 3 (rerun since i didnt save them)
from sklearn.cluster import KMeans

kmeans=KMeans(n_clusters=2,
              random_state=0,
              n_init="auto")
cluster_labels=kmeans.fit_predict(X)

### SVM

In [5]:
X

Gene,ENSG00000203618,ENSG00000229807,ENSG00000285505,ENSG00000266844,ENSG00000243509,ENSG00000176868,ENSG00000105695,ENSG00000079215,ENSG00000168314,ENSG00000091513,...,ENSG00000204481,ENSG00000125823,ENSG00000184900,ENSG00000155858,ENSG00000160145,ENSG00000165186,ENSG00000182511,ENSG00000187609,ENSG00000065978,ENSG00000165209
X1834_AnCg_C_SL31501,2.322181,1.284973,2.049782,1.222389,1.371769,2.241027,5.164592,5.406380,5.450045,5.323159,...,-0.071428,0.396681,3.038149,2.273346,4.628198,2.361969,1.653341,2.855754,2.868216,2.543485
X1881_AnCg_B_SL7146,2.484571,1.184677,1.008067,2.371300,-0.135615,2.398891,4.878538,5.109119,5.114278,5.235119,...,0.464527,0.391462,2.930449,2.257021,4.510377,2.331056,1.660611,2.575238,2.778276,2.964625
X2208_AnCg_M_SL7409,2.063492,3.616451,1.002395,2.796490,-0.108669,2.231441,2.892834,3.749926,5.206875,3.591263,...,0.402256,0.366485,3.110584,2.390957,4.569567,2.388241,1.639343,2.523637,2.803582,3.028239
X2267_AnCg_M_SL7388,-0.074552,1.211657,1.273797,1.567487,-0.074552,2.577255,3.067695,3.213533,4.233820,4.042648,...,-0.074552,0.326445,3.125764,2.568810,4.744089,2.487182,1.649991,2.157021,2.803582,3.409731
X2292_AnCg_C_SL6660,2.048814,1.246053,2.995427,1.657259,-0.091056,1.697494,4.182453,3.946608,4.677841,5.104404,...,0.229314,0.428893,2.878751,2.637429,4.694712,2.520458,1.652309,2.635947,2.599513,3.157553
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
X4413_AnCg_S_SL7398,1.906446,1.168069,-0.101109,3.081055,2.026101,2.322345,5.244080,4.037621,6.215288,5.454258,...,-0.101109,-0.101109,2.790163,2.163069,4.444084,2.281571,1.779006,2.824437,3.081411,2.748427
X4463_AnCg_S_SL6616,-0.183363,1.464153,-0.183363,3.482691,2.331383,2.937264,3.491673,6.290208,6.115600,4.284938,...,0.500144,0.465724,2.374181,1.777299,4.642669,2.762655,2.440505,2.676803,3.147206,2.992206
X4619_AnCg_S_SL6614,-0.107375,1.169112,1.986181,3.028847,-0.107375,1.982674,4.823005,5.619595,6.325133,5.271088,...,0.249174,0.169288,2.738174,2.301376,4.613868,2.410522,1.580039,2.368713,2.953099,2.852084
X4639_AnCg_B_SL7702,2.468156,1.543956,-0.138825,3.618123,1.640628,2.941383,3.920029,5.555397,6.908022,4.770610,...,0.137179,0.142634,2.465046,1.906311,4.400274,2.450456,1.752002,2.568628,3.131295,2.923046


In [5]:
#fitting on labels from assignment 1
model_label=svm.SVC()
model_label.fit(X, labels)
label_predictions=model_label.predict(X)

In [6]:
#fitting on labels from assignment 3
model_cluster=svm.SVC()
model_cluster.fit(X, cluster_labels)
cluster_predictions=model_cluster.predict(X)

In [26]:
svm_results=[label_predictions, cluster_predictions]
svm_results_pdf=pd.DataFrame(svm_results)
svm_results_pdf.to_csv("svm_results_labels.tsv", sep="\t", index=False)

### calculating sample specific area under ROC curve

### retraining models

In [7]:
#retrain model_label

n_genes=[10, 100, 1000, 10000, 5000]
results_labels=[]
aucs_labels=[]
for n in n_genes:
    top = log_expr.var(axis=1).sort_values(ascending=False).index[:n]
    X = log_expr.loc[top].T

    model=svm.SVC()
    model.fit(X, labels)
    label_predictions=model.predict(X)
    results_labels.append(label_predictions)

    #get the auc
    decision_scores=model.decision_function(X)
    auc_score=roc_auc_score(labels, decision_scores)
    aucs_labels.append(auc_score)

In [8]:
# predicted_labels = your list of four arrays
counts = pd.DataFrame(
    [
        {
            "Disorder": np.count_nonzero(labels == "Disorder"),
            "Control": np.count_nonzero(labels == "Control"),
        }
        for labels in results_labels
    ],
    index=[f"model_{i + 1}" for i in range(len(results_labels))]
)

print(counts)

         Disorder  Control
model_1        89        0
model_2        89        0
model_3        89        0
model_4        89        0
model_5        89        0


In [9]:
aucs_labels

[0.962686567164179, 0.9579375848032564, 0.9993215739484397, 1.0, 1.0]

In [11]:
#retrain model_cluster

n_genes=[10, 100, 1000, 10000, 5000]
results_clusters=[]
aucs_clusters=[]
for n in n_genes:
    top = log_expr.var(axis=1).sort_values(ascending=False).index[:n]
    X = log_expr.loc[top].T

    model=svm.SVC()
    model.fit(X, cluster_labels)
    label_predictions=model.predict(X)
    results_clusters.append(label_predictions)

    #get the auc
    decision_scores=model.decision_function(X)
    auc_score=roc_auc_score(cluster_labels, decision_scores)
    aucs_clusters.append(auc_score)

In [12]:
counts = pd.DataFrame(
    [
        {
            "0": np.count_nonzero(labels == 0),
            "1": np.count_nonzero(labels == 1),
        }
        for labels in results_clusters
    ],
    index=[f"model_{i + 1}" for i in range(len(results_clusters))]
)

print(counts)

          0   1
model_1  78  11
model_2  79  10
model_3  79  10
model_4  84   5
model_5  82   7


In [13]:
aucs_clusters

[0.9904761904761904, 0.9990476190476191, 1.0, 1.0, 1.0]

### Heatmaps and dendograms